In [1]:
import json
import numpy as np
from pathlib import Path

BASE_DIR = Path("human_scores")

AUDITOR_DIMS = [
    "issue_identification",
    "reasoning_quality",
    "recommendation_actionability",
    "rubric_calibration",
    "overall_audit_quality"
]

ARCHITECT_DIMS = [
    "contract_alignment",
    "fix_report_accuracy",
    "architecture_quality",
    "security_coverage",
    "plan_completeness"
]

def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def compute_averages(records, dims):
    scores = {d: [] for d in dims}
    for r in records:
        for d in dims:
            if d in r.get("scores", {}) and isinstance(r["scores"][d], (int, float)):
                scores[d].append(float(r["scores"][d]))
    return {d: float(np.mean(scores[d])) if scores[d] else None for d in dims}

def process_user(user_path):
    results = {}

    auditor_file = user_path / "auditor_scores.jsonl"
    architect_file = user_path / "architect_scores.jsonl"

    if auditor_file.exists():
        auditor_records = load_jsonl(auditor_file)
        results["auditor"] = compute_averages(auditor_records, AUDITOR_DIMS)

    if architect_file.exists():
        architect_records = load_jsonl(architect_file)
        results["architect"] = compute_averages(architect_records, ARCHITECT_DIMS)

    return results

all_users_results = {}

for user_folder in BASE_DIR.iterdir():
    if user_folder.is_dir():
        user_name = user_folder.name
        user_results = process_user(user_folder)
        if user_results:
            all_users_results[user_name] = user_results

print("\n" + "="*60)
print("HUMAN EVALUATION RESULTS")
print("="*60)

for user, res in all_users_results.items():
    print(f"\nUser: {user}")
    
    if "auditor" in res:
        print("\n  AUDITOR:")
        for dim, val in res["auditor"].items():
            if val is not None:
                bar = "#" * int(val * 3)
                print(f"    {dim:35s}: {val:.2f}/10  {bar}")
    
    if "architect" in res:
        print("\n  ARCHITECT:")
        for dim, val in res["architect"].items():
            if val is not None:
                bar = "#" * int(val * 3)
                print(f"    {dim:35s}: {val:.2f}/10  {bar}")

output = {
    user: res for user, res in all_users_results.items()
}

with open("human_eval_averages.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\nSaved: human_eval_averages.json")


HUMAN EVALUATION RESULTS

User: Hashan

  AUDITOR:
    issue_identification               : 9.24/10  ###########################
    reasoning_quality                  : 8.45/10  #########################
    recommendation_actionability       : 8.55/10  #########################
    rubric_calibration                 : 7.22/10  #####################
    overall_audit_quality              : 8.08/10  ########################

  ARCHITECT:
    contract_alignment                 : 7.98/10  #######################
    fix_report_accuracy                : 8.47/10  #########################
    architecture_quality               : 7.51/10  ######################
    security_coverage                  : 7.24/10  #####################
    plan_completeness                  : 8.51/10  #########################

Saved: human_eval_averages.json
